# Comparación NPP-LSTM frente a Poisson

## Objetivo

Comparar el modelo NPP-LSTM Softplus W20 seleccionado
por validación con el proceso de Poisson implementado
en el proyecto original.

Horizontes: 1, 7 y 30 días.

Métricas:
- Brier Score
- Log Loss
- Frecuencia observada
- Probabilidad media estimada

Esta es una evaluación histórica retrospectiva.
No constituye un sistema operativo de alerta sísmica.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

raiz = Path.cwd()
if not (raiz / "data").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

from utils import (
    datetime_to_hours,
    truncate_catalog_by_threshold,
    append_burn_in_to_test_set,
    find_Poisson_MLE
)
from npp_model import construir_npp
from npp_training import preparar_ventanas

VENTANA = 20
MCUT = 3.5
M0PRED = 4.5

catalogo = pd.read_csv(
    raiz / "data/raw/SISMOS_LIMA_FORMATO_MODELO_cat.csv"
)
catalogo = datetime_to_hours(catalogo).dropna()
catalogo = truncate_catalog_by_threshold(catalogo, MCUT)

tiempos = catalogo["time"].to_numpy()
magnitudes = catalogo["mw"].to_numpy()

T_train = tiempos[tiempos < 450000]
M_train = magnitudes[tiempos < 450000]

T_test = tiempos[tiempos >= 450000]
M_test = magnitudes[tiempos >= 450000]

# Poisson original
tasa_poisson = find_Poisson_MLE(
    T_train, M_train, M0PRED
)

# Normalización usada al entrenar W20
mu_t = np.log(np.diff(T_train)).mean()
sigma_t = np.log(np.diff(T_train)).std()
mu_m = np.log(M_train).mean()
sigma_m = np.log(M_train).std()

modelo = construir_npp(
    ventana=VENTANA,
    mu_t=mu_t,
    sigma_t=sigma_t,
    mu_m=mu_m,
    sigma_m=sigma_m,
    m0pred=M0PRED,
    activacion_magnitud="softplus"
)

modelo.load_weights(
    str(raiz / "models/NPP_softplus_W20.weights.h5")
)

T_eval, M_eval = append_burn_in_to_test_set(
    T_train, T_test, M_train, M_test, VENTANA
)

X_eval = preparar_ventanas(
    T_eval, M_eval, VENTANA, M0PRED
)

# Cada historial termina en esta fecha de referencia
referencias = T_eval[VENTANA:-1]

assert len(referencias) == len(X_eval[0])

print("Modelo NPP Softplus W20 recuperado.")
print("Tasa Poisson por hora:", tasa_poisson)
print("Referencias históricas:", len(referencias))


Modelo NPP Softplus W20 recuperado.
Tasa Poisson por hora: 0.003581180843539231
Referencias históricas: 818


In [2]:
def probabilidad_npp(horas):
    n = len(X_eval[0])

    hist_t = tf.convert_to_tensor(X_eval[0])
    hist_m = tf.convert_to_tensor(X_eval[1])

    # Magnitud auxiliar: la salida H no depende de ella.
    mag = tf.fill((n, 1), tf.constant(4.5, tf.float32))

    # Aproximación a tiempo cero: log(0) no está definido.
    dt_inicio = tf.fill(
        (n, 1), tf.constant(1e-3, tf.float32)
    )

    dt_horizonte = tf.fill(
        (n, 1), tf.constant(float(horas), tf.float32)
    )

    H_inicio, _ = modelo(
        [hist_t, hist_m, dt_inicio, mag],
        training=False
    )

    H_final, _ = modelo(
        [hist_t, hist_m, dt_horizonte, mag],
        training=False
    )

    delta_H = (
        H_final.numpy().ravel()
        - H_inicio.numpy().ravel()
    )

    delta_H = np.maximum(delta_H, 0.0)

    return np.clip(
        -np.expm1(-delta_H), 0.0, 1.0
    )


def probabilidad_poisson(horas):
    return -np.expm1(-tasa_poisson * horas)


# Todos los eventos observados que alcanzan Mw 4.5
eventos_objetivo = tiempos[magnitudes >= M0PRED]


def ocurrencia_observada(horas):
    # Eventos posteriores a cada fecha de referencia
    inicio = np.searchsorted(
        eventos_objetivo, referencias, side="right"
    )

    fin = np.searchsorted(
        eventos_objetivo,
        referencias + horas,
        side="right"
    )

    return (fin > inicio).astype(int)

In [3]:
resumen = []
predicciones = []

for dias in [1, 7, 30]:
    horas = dias * 24

    # No evaluar horizontes que superen el fin del catálogo
    validos = referencias + horas <= tiempos[-1]

    y = ocurrencia_observada(horas)[validos]

    p_npp = probabilidad_npp(horas)[validos]

    p_poisson = np.full(
        len(y),
        probabilidad_poisson(horas)
    )

    for nombre, p in [
        ("NPP Softplus W20", p_npp),
        ("Poisson", p_poisson)
    ]:
        p_segura = np.clip(p, 1e-7, 1 - 1e-7)

        brier = np.mean((p - y) ** 2)

        logloss = -np.mean(
            y * np.log(p_segura)
            + (1 - y) * np.log(1 - p_segura)
        )

        resumen.append({
            "Horizonte_dias": dias,
            "Modelo": nombre,
            "Muestras": len(y),
            "Frecuencia_observada": y.mean(),
            "Probabilidad_media": p.mean(),
            "Brier": brier,
            "LogLoss": logloss
        })

        predicciones.extend(
            {
                "Horizonte_dias": dias,
                "Modelo": nombre,
                "Referencia_horas": float(t),
                "Observado": int(obs),
                "Probabilidad": float(prob)
            }
            for t, obs, prob in zip(
                referencias[validos], y, p
            )
        )

resultados = pd.DataFrame(resumen)

display(resultados.round(4))

(raiz / "results").mkdir(exist_ok=True)

resultados.to_csv(
    raiz / "results/comparacion_NPP_Poisson.csv",
    index=False
)

pd.DataFrame(predicciones).to_csv(
    raiz / "results/predicciones_NPP_Poisson.csv",
    index=False
)

,Horizonte_dias,Modelo,Muestras,Frecuencia_observada,Probabilidad_media,Brier,LogLoss
0,1,NPP Softplus W20,818,0.0856,0.1596,0.0805,0.3008
1,1,Poisson,818,0.0856,0.0824,0.0783,0.2922
2,7,NPP Softplus W20,818,0.3778,0.6332,0.2871,0.7722
3,7,Poisson,818,0.3778,0.4521,0.2406,0.6743
4,30,NPP Softplus W20,813,0.7565,0.9684,0.2289,0.8992
5,30,Poisson,813,0.7565,0.9241,0.2123,0.6877
